# Grasp-Anything++ — train, eval, và nhìn text align vào đâu

Notebook chạy trọn một vòng trên máy có GPU (Colab / Kaggle / vast.ai):

1. clone repo, cài dependency
2. dựng **subset** GA++ (~10.6 GB tải về, không phải 150 GB)
3. dựng split seen/unseen theo protocol paper LGD
4. train `grconvnet3_align`
5. eval → bảng Seen / Unseen / **H**
6. **visualize**: token nào trong prompt đang align với vùng nào của ảnh

Bước 6 mới là thứ đáng xem — nó cho thấy nhánh ngôn ngữ có thật sự chọn *vùng* hay chỉ là
một vector bị nhét vào feature.

> Cần: GPU (T4 là đủ), ~20 GB đĩa trống. Bước 2 là bước lâu nhất.

## 0. Môi trường

In [ ]:
import os
import shutil
import subprocess
import sys

print("python  :", sys.version.split()[0])
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True)
    print("gpu     :", out.stdout.strip() or "không thấy — sẽ chạy CPU, rất chậm")
except FileNotFoundError:
    print("gpu     : không có nvidia-smi — sẽ chạy CPU, rất chậm")
total, used, free = shutil.disk_usage(".")
print(f"đĩa     : {free / 1e9:.0f} GB trống")

In [ ]:
REPO_URL = "https://github.com/duncan-nguyen/QACI-HW.git"
BRANCH = "text-image-aware"
REPO_DIR = "QACI-HW"


def run(cmd, **kw):
    """Chạy lệnh, in output ngay khi có (subprocess thay vì ! để notebook chạy được ở mọi kernel)."""
    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"lệnh lỗi (mã {proc.returncode})")


def in_repo():
    """Đang đứng sẵn trong bản checkout của repo (chạy notebook từ trong repo)?"""
    return os.path.isdir(".git") and os.path.basename(os.getcwd()) == REPO_DIR


if in_repo():
    print("đã ở trong repo, không clone")
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print("dùng bản clone có sẵn")
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)

dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True,
                       text=True).stdout.strip()
if dirty:
    # Có thay đổi chưa commit -> không đụng vào, tránh nuốt mất việc đang làm dở.
    print("có thay đổi chưa commit, bỏ qua checkout/pull:")
    print(dirty)
else:
    run(["git", "checkout", BRANCH])
    run(["git", "pull", "--ff-only"])

sys.path.insert(0, os.getcwd())
print("\ncwd:", os.getcwd())

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
# pyrealsense2 chỉ cần cho robot thật; thiếu nó không ảnh hưởng train/eval.
run([sys.executable, "-c", "import torch, transformers, skimage, cv2; "
     "print('torch', torch.__version__, '| cuda', torch.cuda.is_available())"])

## 1. Cấu hình

In [ ]:
DATA_DIR = "data/ga-pp-subset"
SPLIT_DIR = "split/grasp-anything-pp"

# Mặc định dưới đây nhắm Colab Pro / A100 80GB (12 vCPU, ~166 GB đĩa).
#
# GR-ConvNet ở 224x224 rất nhẹ, A100 80GB không bao giờ là chỗ nghẽn. Nghẽn nằm ở **CPU**:
# mỗi sample phải decode JPEG, rotate/zoom ảnh + part_mask, và rasterize M_union từ mọi part
# của object. Nên tăng NUM_WORKERS quan trọng hơn tăng BATCH_SIZE. Cell benchmark ở §4.1 đo
# giúp bạn đang bị nghẽn ở đâu.
#
# Đĩa: part_mask là 173 KB/sample chưa nén và chiếm gần hết dung lượng.
#   2.000 scene ≈  8,8k sample ≈  1,6 GB
#  10.000 scene ≈  44k  sample ≈  7,8 GB   <- mặc định
#  20.000 scene ≈  88k  sample ≈   16 GB

N_SCENES = 10_000        # ~4.4 sample/scene
EPOCHS = 30
BATCHES_PER_EPOCH = 500
BATCH_SIZE = 64
NUM_WORKERS = 8
VAL_SPLIT = 0.98         # 2% giữ lại làm validation; validate chạy batch_size=1 nên đừng để to
WARMUP_EPOCHS = 3
W_ALIGN = 1.0            # trọng số L_align(A_T, part_mask)
W_AGNOSTIC = 1.0         # trọng số L_agnostic(Q_g, M_union)
DESCRIPTION = "ga-pp-align"

print(f"ngân sách train: {EPOCHS} x {BATCHES_PER_EPOCH} x {BATCH_SIZE} = "
      f"{EPOCHS * BATCHES_PER_EPOCH * BATCH_SIZE:,} sample "
      f"(~{EPOCHS * BATCHES_PER_EPOCH * BATCH_SIZE / (N_SCENES * 4.4):.1f} lượt qua subset)")

## 2. Dựng subset

GA++ đủ bộ là 4,4 triệu sample / 150 GB. `script/build_ga_pp_subset.py` tải 4 zip label
(~10.6 GB) rồi chỉ trích ra các scene được chọn, còn ảnh thì đọc từng file cần qua HTTP range
nên **không** phải tải archive ảnh 65 GB.

Chọn theo **scene**, không phải theo sample: `M_∪` cần mọi part của cùng một object, lấy sample
rời rạc thì union suy biến thành chính part đó.

In [ ]:
run([sys.executable, "script/build_ga_pp_subset.py",
     "--out", DATA_DIR, "--scenes", N_SCENES, "--workers", 32])

## 3. Split seen / unseen

Theo paper §5.1: chia theo **category object** — 70% category theo tần suất giảm dần vào Base
(seen), 30% còn lại vào New (unseen). Category lấy từ `scene_description`.

In [ ]:
run([sys.executable, "split/build_grasp_anything_pp.py",
     "--data-dir", DATA_DIR, "--out-dir", SPLIT_DIR])

In [ ]:
from utils.data import get_dataset

Dataset = get_dataset("grasp-anything-pp")
for seen in (True, False):
    ds = Dataset(DATA_DIR, seen=seen, split_path=SPLIT_DIR, output_size=224,
                 include_depth=False, include_rgb=True)
    parts = [len(v) for v in ds._files_by_object.values()]
    print(f"{'seen  ' if seen else 'unseen'}: {len(ds):>6,} sample, "
          f"{len(ds._files_by_object):>6,} object, "
          f"part/object trung bình {sum(parts) / len(parts):.2f}")
    if seen:
        x, y, idx, rot, zoom, extra = ds[0]
        print(f"         mẫu: {extra['prompt']!r}  part_mask fg={extra['part_mask'].mean() * 100:.1f}%")

## 4. Train

Train trên tập **seen**; trong đó `--split 0.9` giữ lại 10% cuối làm validation. Tập unseen
không được chạm tới lúc train — nó là category chưa từng thấy.

### 4.1 Đo tốc độ loader trước khi train

Chạy 30 batch để biết đang nghẽn ở CPU hay không. Nếu throughput thấp hơn ~`NUM_WORKERS x 40`
sample/s thì tăng `NUM_WORKERS`; tăng `BATCH_SIZE` sẽ không giúp gì.

In [ ]:
import time

import torch

bench_ds = Dataset(DATA_DIR, seen=True, split_path=SPLIT_DIR, output_size=224,
                   include_depth=False, include_rgb=True, random_rotate=True, random_zoom=True)
bench_dl = torch.utils.data.DataLoader(
    bench_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS else None,
    pin_memory=True)

N_BENCH = 30
t0, n, nb = None, 0, 0
for batch in bench_dl:
    if t0 is None:                          # bỏ batch đầu: nó gánh chi phí spawn worker
        t0 = time.time()
        continue
    n += batch[0].shape[0]
    nb += 1
    if nb >= N_BENCH:
        break

dt = time.time() - t0
rate = n / dt if dt > 0 else float("nan")
print(f"{rate:.0f} sample/s  ({dt / max(nb, 1) * 1000:.0f} ms/batch, batch={BATCH_SIZE}, "
      f"workers={NUM_WORKERS}, đo trên {nb} batch)")
print(f"-> một epoch {BATCHES_PER_EPOCH} batch mất khoảng "
      f"{BATCHES_PER_EPOCH * BATCH_SIZE / rate / 60:.1f} phút nếu nghẽn ở loader")
del bench_dl, bench_ds

### 4.2 Chạy train

In [ ]:
run([sys.executable, "train_network.py",
     "--dataset", "grasp-anything-pp", "--dataset-path", DATA_DIR,
     "--split-path", SPLIT_DIR, "--network", "grconvnet3_align",
     "--use-depth", 0, "--use-rgb", 1, "--seen", 1,
     "--input-size", 224, "--split", VAL_SPLIT,
     "--epochs", EPOCHS, "--batches-per-epoch", BATCHES_PER_EPOCH,
     "--batch-size", BATCH_SIZE, "--num-workers", NUM_WORKERS,
     "--use-text", 1, "--w-align", W_ALIGN, "--w-agnostic", W_AGNOSTIC,
     "--warmup-epochs", WARMUP_EPOCHS,
     "--logdir", "logs/", "--description", DESCRIPTION])

In [ ]:
import glob
import re


def best_checkpoint(description):
    """Checkpoint có IoU validation cao nhất của lần chạy gần nhất."""
    runs = sorted(glob.glob(os.path.join("logs", f"*{description}*")))
    if not runs:
        raise FileNotFoundError("không thấy thư mục log nào")
    ckpts = glob.glob(os.path.join(runs[-1], "epoch_*"))
    if not ckpts:
        raise FileNotFoundError(f"không thấy checkpoint trong {runs[-1]}")
    return max(ckpts, key=lambda p: float(re.search(r"iou_([0-9.]+)$", p).group(1)))


CHECKPOINT = best_checkpoint(DESCRIPTION)
print("checkpoint:", CHECKPOINT, f"({os.path.getsize(CHECKPOINT) / 1e6:.0f} MB)")

## 5. Eval — Seen / Unseen / H

Metric của paper: success khi IoU ≥ 0.25 **và** lệch góc ≤ 30°.

- **seen**: `--split VAL_SPLIT`, đúng phần đã giữ lại lúc train.
- **unseen**: `--split 0.0`, toàn bộ — model chưa từng thấy category này.

In [ ]:
def evaluate(checkpoint, seen, split):
    cmd = [sys.executable, "evaluate.py",
           "--dataset", "grasp-anything-pp", "--dataset-path", DATA_DIR,
           "--split-path", SPLIT_DIR, "--network", checkpoint, "--iou-eval",
           "--use-depth", "0", "--use-rgb", "1",
           "--seen", str(int(seen)), "--split", str(split),
           "--num-workers", str(NUM_WORKERS)]
    print("$", " ".join(cmd))
    out = subprocess.run(cmd, capture_output=True, text=True)
    text = out.stdout + out.stderr
    hit = re.findall(r"IOU Results: \d+/\d+ = ([0-9.]+)", text)
    if not hit:
        print(text[-2000:])
        raise RuntimeError("evaluate.py không in ra kết quả IoU")
    return float(hit[-1])


acc_seen = evaluate(CHECKPOINT, seen=True, split=VAL_SPLIT)
acc_unseen = evaluate(CHECKPOINT, seen=False, split=0.0)
harmonic = 0.0 if acc_seen + acc_unseen == 0 else 2 * acc_seen * acc_unseen / (acc_seen + acc_unseen)

print()
print(f"{'':22} {'Seen':>7} {'Unseen':>7} {'H':>7}")
print(f"{'ours (align)':22} {acc_seen:7.3f} {acc_unseen:7.3f} {harmonic:7.3f}")
print(f"{'GR-ConvNet+CLIP †':22} {0.37:7.2f} {0.18:7.2f} {0.24:7.2f}")
print(f"{'CLIP-Fusion †':22} {0.40:7.2f} {0.29:7.2f} {0.33:7.2f}")
print(f"{'LGD †':22} {0.48:7.2f} {0.42:7.2f} {0.45:7.2f}")
print("\n† Table 2 của paper, train trên GA++ đầy đủ — để đối chiếu, không so ngang được với")
print("  subset nhỏ ở đây.")

## 6. Text align vào đâu?

Phần chính. `A_T(x, y) = σ(max_j cos(W_v F_xy, W_t t_j) / τ)` được tính ở bottleneck 56×56, và
`per_token` giữ lại similarity của **từng** token trước khi lấy max — nên nhìn được token nào
kéo attention về vùng nào.

In [ ]:
import matplotlib.pyplot as plt
import torch

from utils.visualisation.alignment import (plot_part_prompts, plot_prompt_comparison,
                                           plot_token_alignment)

device = "cuda" if torch.cuda.is_available() else "cpu"
net = torch.load(CHECKPOINT, map_location=device, weights_only=False).eval()

ds_seen = Dataset(DATA_DIR, seen=True, split_path=SPLIT_DIR, output_size=224,
                  include_depth=False, include_rgb=True)
ds_unseen = Dataset(DATA_DIR, seen=False, split_path=SPLIT_DIR, output_size=224,
                    include_depth=False, include_rgb=True)
print(f"seen {len(ds_seen):,} | unseen {len(ds_unseen):,} | device {device}")

### 6.1 Từng token của prompt

Mỗi ô là bản đồ similarity của một token, chuẩn hoá chung trong cùng một sample nên so được
với nhau. Số trong ngoặc là activation cực đại của token đó; token được sắp giảm dần.

Kỳ vọng: danh từ part (`handle`, `rim`, `cap`, `skin`) sáng lên đúng vùng của `part_mask`, còn
động từ (`grasp`, `pick`) và giới từ thì tản đều — chúng không mang thông tin vị trí.

In [ ]:
for idx in [0, len(ds_seen) // 3, 2 * len(ds_seen) // 3]:
    x, _, _, _, _, extra = ds_seen[idx]
    fig, ranked = plot_token_alignment(net, x, extra["prompt"],
                                       part_mask=extra["part_mask"], max_tokens=7)
    plt.show()
    print("  " + "  ".join(f"{t}={v:.2f}" for t, v in ranked))

### 6.2 Cùng ảnh, đổi part trong câu lệnh

Ảnh không đổi, chỉ đổi part được nhắc tới. Nếu `A_T` dịch theo `part_mask` của từng hàng thì
nhánh ngôn ngữ đang làm đúng việc của nó.

In [ ]:
# Chọn object có nhiều part nhất để nhìn cho rõ.
object_id, files = max(ds_seen._files_by_object.items(), key=lambda kv: len(kv[1]))
indices = [ds_seen.grasp_files.index(f) for f in files]
print(f"object {object_id[:16]}… có {len(indices)} part")

fig = plot_part_prompts(net, ds_seen, indices[:4])
plt.show()

### 6.3 Prompt tự viết

Không giới hạn ở prompt có trong dataset — gõ câu bất kỳ và xem `A_T` phản ứng. Đây là chỗ dễ
thấy nhất khi model *không* học được gì: mọi prompt cho ra cùng một bản đồ.

In [ ]:
x, _, _, _, _, extra = ds_unseen[0]
print("prompt gốc:", repr(extra["prompt"]))

fig = plot_prompt_comparison(net, x, [
    extra["prompt"],
    "Grasp the object at its handle.",
    "Grasp the object at its top.",
    "Grasp the object at its bottom.",
])
plt.show()

### 6.4 Trên category chưa từng thấy

Cùng cách nhìn nhưng ở tập unseen. Từ vựng part (`handle`, `cap`, `rim`) dùng chung giữa các
category, nên nếu grounding học được thật thì nó vẫn phải chỉ đúng chỗ dù object là mới.

In [ ]:
for idx in [0, len(ds_unseen) // 2]:
    x, _, _, _, _, extra = ds_unseen[idx]
    fig, ranked = plot_token_alignment(net, x, extra["prompt"],
                                       part_mask=extra["part_mask"], max_tokens=7)
    plt.show()
    print("  " + "  ".join(f"{t}={v:.2f}" for t, v in ranked))

## 7. (tuỳ chọn) Arm baseline để có bảng ablation

Cùng file model, tắt nhánh ngôn ngữ. Giữ **nguyên** ngân sách train để so được.
Xem §6 của `idea.md` cho đủ bốn arm.

In [ ]:
RUN_BASELINE = False

if RUN_BASELINE:
    run([sys.executable, "train_network.py",
         "--dataset", "grasp-anything-pp", "--dataset-path", DATA_DIR,
         "--split-path", SPLIT_DIR, "--network", "grconvnet3_align",
         "--use-depth", 0, "--use-rgb", 1, "--seen", 1,
         "--input-size", 224, "--split", VAL_SPLIT,
         "--epochs", EPOCHS, "--batches-per-epoch", BATCHES_PER_EPOCH,
         "--batch-size", BATCH_SIZE, "--num-workers", NUM_WORKERS,
         "--use-text", 0, "--w-agnostic", 0,
         "--logdir", "logs/", "--description", "ga-pp-notext"])

    ckpt = best_checkpoint("ga-pp-notext")
    s = evaluate(ckpt, seen=True, split=VAL_SPLIT)
    u = evaluate(ckpt, seen=False, split=0.0)
    h = 0.0 if s + u == 0 else 2 * s * u / (s + u)
    print(f"\n{'no-text baseline':22} {s:7.3f} {u:7.3f} {h:7.3f}")
else:
    print("RUN_BASELINE = False")

## Ghi chú

- Subset ~2.000 scene là cỡ Cornell (~1.000 ảnh). Số tuyệt đối sẽ thấp hơn Table 2 của paper
  (train trên 4,4M sample) — dùng để so giữa các arm với nhau, đừng so thẳng với paper.
- Checkpoint ~265 MB vì `torch.save(net)` pickle cả CLIP text tower; đổi lại load ra là chạy
  được ngay, không cần dựng lại kiến trúc.
- Nếu `A_T` trông như nhiễu ở mọi prompt: kiểm tra `align_loss` trong log tensorboard có giảm
  không. Không giảm thì thử tăng `W_ALIGN` hoặc kéo dài `WARMUP_EPOCHS`.
- Muốn nét hơn ở mức part: `F` đang là 56×56 (stride 4). Tính `A_T` sau `conv4` (112×112) sẽ
  mịn hơn, đổi lại tốn bộ nhớ.